# PROC GLMSELECTで需要ドライバーを特定する：段階的選択、LASSO、検証付き前進選択

## エグゼクティブサマリー

小売分析チームは**PROC GLMSELECT**を使い、実際に週間販売数量を動かしているマーケティング・価格施策を特定し、真の需要ドライバーをノイズから切り分ける。SBCで評価する段階的選択、5分割交差検証によるLASSO、ホールドアウト検証付きの前進選択のいずれもが**同じ真のドライバー集合**——自社価格、競合価格、広告費、メール送信数、販促、祝日、Northeast地域、Onlineチャネル——を復元し、仕込んだ4つのデコイ変数（`temp_f`、`noise1`、`noise2`、`noise3`）はすべて自動的に除外される。

各手法は効果の大きさについてもほぼ一致する：自社価格効果はいずれも**1ドルあたり約-28ユニット**、競合価格効果は**約+14**と推定され、データ生成式に組み込んだ代替効果の符号と一致する。唯一の相違点は境界線上にある——交差検証によるLASSOのみが、統計的にはごくわずかな`Region=Midwest`のコントラスト（推定値-8.3、標準誤差8.3）を残す一方、段階的選択と前進選択はどちらもこれを除外する。3つの異なる選択方針すべてを生き残ったドライバーリストは、単一の基準に合わせて調整したものよりもはるかに信頼できる。

## データソース

このノートブックのデータはすべて**合成データ**であり、インラインで生成される（外部ファイルなし、シード`20260531`）。消費財小売業者の店舗×週次需要パネル1シーズン分を模擬している。

| データセット | 行数 | 粒度 | 主な変数 |
|---------|------|------|---------------|
| `demand` | 100 | 店舗×週 | `units`（応答変数：週間販売数量）；`price`、`competitor`（自社・競合の店頭価格）；`adspend`、`email`（メディア露出）；`promo`、`holiday`（イベントフラグ）；`region`、`channel`（CLASS効果）；`temp_f`、`noise1`〜`noise3`（デコイ／無関係な説明変数） |

`units`は既知の線形方程式から生成されるため、「正しい」ドライバー集合は検証可能である。`temp_f`と3つの`noise`列は真のシグナルを持たず、各選択手法がそれらを除外できるかを試すために存在する。

# PROC GLMSELECTで需要ドライバーを特定する

カテゴリーマネージャーには週間売上を説明しうる候補が何十個もある：自社価格、競合価格、広告費、メール送信数、販促、祝日、店舗地域、販売チャネル、さらには天候まで。それらすべてを一つの回帰に投入すると過学習と係数の不安定化を招く。**PROC GLMSELECT**は倹約なモデルの探索を自動化し、段階的選択、LASSO、エラスティックネット、最小角回帰選択を交差検証とホールドアウト分割込みでサポートする。

このノートブックでは：

1. *既知の*真のドライバー集合（および意図的なデコイ変数）を持つ、現実的な合成の店舗×週次需要パネルを生成する。
2. SBCで評価する**段階的選択**を実行する。
3. 5分割交差検証による**LASSO**を実行する。
4. **30%ホールドアウトで検証した前進選択**を実行する。

優れた選択手法であれば、真のドライバーを復元しノイズを除外できるはずだ——確認してみよう。

## 1. 合成需要パネルを生成する

応答変数`units`は明示的な線形方程式から構成されるため、真のドライバーは既知である：価格と競合価格、広告費、メール送信数、販促・祝日フラグ、さらに地域とチャネルの効果がすべて寄与する。変数`temp_f`、`noise1`、`noise2`、`noise3`は売上と真の関係を持たない純粋なデコイである。`call streaminit`のシードによりデータは再現可能になる。

In [1]:
/* ---------------------------------------------------------------
   Synthetic store-week demand panel for a consumer-goods retailer.
   'units' follows a KNOWN equation; temp_f and noise1-3 are decoys.
   --------------------------------------------------------------- */
データ demand;
    呼出 streaminit(20260531);
    長さ region $9 channel $8 promo $3;
    繰返 store_week = 1 から 100;
        /* Region mix */
        u1 = rand('uniform');
        もし u1 < 0.34 なら region = 'Northeast';
        他 もし u1 < 0.67 なら region = 'Midwest';
        他 region = 'South';

        /* Sales channel */
        もし rand('uniform') < 0.45 なら channel = 'Store';
        他 channel = 'Online';

        /* Pricing and media drivers */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Event flags and an irrelevant weather decoy */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        もし rand('uniform') < 0.40 なら promo = 'Yes';
        他 promo = 'No';

        /* Pure noise predictors (no true effect) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Weekly units sold from a known structural equation */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        もし units < 0 なら units = 0;
        出力;
    終了;
    保持 region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
実行;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.08 seconds
  cpu   0.08 seconds


## 2. データをプロファイルする

モデリングの前に、応答変数と主な連続変数候補が妥当なスケールにあることを確認する。

In [2]:
処理 平均 データ=demand n mean std MIN MAX maxdec=1;
    変数 units price competitor adspend email;
    見出 units="週間販売数量" price="自社価格" competitor="競合価格" adspend="広告費" email="メール送信数";
    表題 "週次需要と候補ドライバー";
実行;


                                                      週次需要と候補ドライバー                                                      

                                                  The MEANS Procedure

 Variable    Label                      N        Mean     Std Dev     Minimum     Maximum
 ----------------------------------------------------------------------------------------
 units       週間販売数量                   100       874.8       136.3       508.6      1162.3
 price       自社価格                     100        14.0         3.4         8.0        19.9
 competitor  競合価格                     100        13.8         3.4         8.1        19.9
 adspend     広告費                      100       990.6       726.9        50.0      3358.0
 email       メール送信数                   100        45.5        26.4         0.0        99.0
 ----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. SBCで評価する段階的選択

段階的選択は効果の追加と除去を交互に行う手法で、ここでは投入判定（`select=sbc`）と最終モデル選択（`choose=sbc`）の両方を**シュワルツ・ベイズ基準（SBC）**で判定する。SBCはAICよりも複雑さへのペナルティが大きく、より簡潔なモデルを選好する。

主なステートメントとオプション：

- `CLASS region channel promo / param=reference`は基準セル符号化でカテゴリカルな説明変数を宣言する。
- `selection=stepwise(select=sbc choose=sbc)`が探索を駆動する。
- `details=summary`はステップごとの選択サマリーを出力し、`stb`は標準化係数を追加してスケールの異なる効果を比較可能にする。
- `output out=demand_scored p=predicted r=residual`は下流でのスコアリング用に予測値と残差を保存する。

**Stepwise Selection Summary**を探索の軌跡として読み解く：フル12効果モデルから出発し、`noise1`、`noise2`、`temp_f`、`Region=Midwest`のコントラスト、`noise3`を順に一つずつ*除去*していく。各除去がSBCを下げるからである。**Parameter Estimates**表に残るものが選ばれたモデルである。

In [3]:
処理 glmselect データ=demand seed=20260531;
    分類 region channel promo / PARAM=reference;
    模型 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    出力 out=demand_scored p=predicted r=residual;
    見出 units="週間販売数量" price="自社価格" competitor="競合価格" adspend="広告費"
          email="メール送信数" region="地域" channel="販売チャネル" promo="販促の有無"
          holiday="祝日フラグ" temp_f="気温(F)" noise1="ノイズ1" noise2="ノイズ2" noise3="ノイズ3";
    表題 "SBCによる需要ドライバーの段階的選択";
実行;


                                                      週次需要と候補ドライバー                                                      

The GLMSELECT Procedure


Dependent Variable: UNITS 週間販売数量


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                Stepwise Selection Summary                                                 

    Step    Action          Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  --------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed            ノイズ1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
       2   Removed  


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. 交差検証によるLASSO

LASSOは係数をゼロに向けて縮小させ、選択と正則化を同時に行う。固定基準で止める代わりに、**5分割交差検証**にLASSOパス上で汎化誤差が最良となる点を選ばせる。

- `selection=lasso(choose=cv stop=none)`はLASSOパス全体をたどり、CV最適なステップを選択する。
- `cvmethod=random(5)`は観測値を5つのランダムな分割に割り当てる。

**LASSO Selection Summary**は、ペナルティが緩むにつれて効果が投入される順序を示す：まず`price`、次に`adspend`、`competitor`、Northeast地域、`email`、`promo`、`holiday`——7つの真のシグナルすべてがどのデコイよりも先に投入される。交差検証は汎化PRESSが最小となるステップを選び、結果の**Parameter Estimates**表には（小さな`Region=Midwest`項を除き）まさに真のドライバーのみが残り、`temp_f`、`noise1`、`noise2`、`noise3`はパスの最後にしか投入されないため除外される。

In [4]:
処理 glmselect データ=demand seed=20260531;
    分類 region channel promo / PARAM=reference;
    模型 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=random(5);
    見出 units="週間販売数量" price="自社価格" competitor="競合価格" adspend="広告費"
          email="メール送信数" region="地域" channel="販売チャネル" promo="販促の有無"
          holiday="祝日フラグ" temp_f="気温(F)" noise1="ノイズ1" noise2="ノイズ2" noise3="ノイズ3";
    表題 "5分割交差検証によるLASSO";
実行;


                                                      週次需要と候補ドライバー                                                      

The GLMSELECT Procedure


Dependent Variable: UNITS 週間販売数量


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                          LASSO Selection Summary                                                           

    Step    Action              Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ------------------  -----------------  ---


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. ホールドアウトで検証した前進選択

3つ目の補完的な戦略：**前進選択**（効果は投入されるのみで除去されない）でモデルを構築するが、独立したホールドアウトサンプルでの性能が最良となる点で止める——過学習を直接的に防ぐ。

- `partition fraction(validate=0.30)`は行の30%をランダムに検証用に確保し、学習70件・検証30件を残す。
- `selection=forward(select=aic choose=validate)`はAICで効果を追加するが、最終モデルは検証サンプルの誤差で選ぶ。

**Partition Fractions**表は70/30分割を確認する。前進選択は検証誤差の改善が止まるまで効果を投入し続け、最終的な**Parameter Estimates**表の8つの効果はまさに真のドライバーであり、4つのデコイは一度も採用されない。異なる原理に基づく3つの手法が同じドライバーに行き着くとき、そのモデルは単一の選択基準の産物である場合よりもはるかに実在の可能性が高い。

In [5]:
処理 glmselect データ=demand seed=20260531;
    分類 region channel promo / PARAM=reference;
    模型 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition FRACTION(validate=0.30);
    見出 units="週間販売数量" price="自社価格" competitor="競合価格" adspend="広告費"
          email="メール送信数" region="地域" channel="販売チャネル" promo="販促の有無"
          holiday="祝日フラグ" temp_f="気温(F)" noise1="ノイズ1" noise2="ノイズ2" noise3="ノイズ3";
    表題 "30%ホールドアウトで検証した前進選択";
実行;


                                                      週次需要と候補ドライバー                                                      

The GLMSELECT Procedure


Dependent Variable: UNITS 週間販売数量


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         Forward Selection Summary                                                          

    Step    Action              Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ------------------  -----


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. 結果の解釈

3つの選択戦略はすべて**同じ真の需要ドライバー集合**を復元し、すべてのデコイを除外する。**Parameter Estimates**表を直接読むと：

- **Price（価格）**が支配的なドライバーである。段階的選択モデルにおける標準化係数は**-0.70**で群を抜いて大きく、生の傾き（スロープ）は1ドルあたり**-27.8**（段階的選択とLASSO）から**-28.6**（前進選択）ユニットの間に収まり、データ生成に用いた-28にほぼ正確に一致する。**Competitor price（競合価格）**はすべての3つのフィットで約**+14.4**と逆方向に需要を動かし、カテゴリーマネージャーが期待する代替行動と一致する。
- **Ad spend（広告費）**（1ドルあたり約+0.062ユニット）と**email volume（メール送信数）**（1送信あたり約+1.2ユニット）はどちらも売上を押し上げ、メディア反応を定量化する。
- **Promotions（販促）**と**holidays（祝日）**が最大のイベント効果を持つ：`Promo=No`のコントラストは販促週に対して約**-51〜-57**、祝日の押し上げ効果は約**+66〜+76**ユニットである。**Northeast地域**（約+49〜+55）と**Onlineチャネル**（約+24〜+32）は正のベースライン効果を持つ。
- 重要な点として、すべてのデコイ——`temp_f`、`noise1`、`noise2`、`noise3`——は段階的選択と前進選択で**除外**され、CV選択されたLASSOモデルからも除外される。各手法が構造的シグナルを復元し、ノイズを無視した。

3つの手法は完全に同一というわけではない：段階的選択（SBC）とホールドアウト検証済み前進選択は同じ8つの効果に落ち着くが、交差検証によるLASSOはさらに小さな`Region=Midwest`のコントラスト（-8.3、標準誤差8.3）を保持する——これは実質的な不一致というよりノイズの限界での差である。段階的SBC、交差検証LASSO、ホールドアウト検証前進選択のすべてを生き残るドライバーリストこそが本当の教訓である：3つの異なる選択方針に頑健な発見は、単一の基準に合わせたものよりもはるかに信頼できる。`OUTPUT OUT=demand_scored`で予測値と残差を捕捉することで、同じワークフローは来四半期の予定価格・販促カレンダーのスコアリングへと自然に拡張できる。